In [2]:
# Import the pytesseract library, a Python wrapper for Google's Tesseract-OCR engine
import pytesseract
# Set the path to the Tesseract OCR executable (required for Windows if not added to system PATH)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [3]:
# pytesseract version 
print(pytesseract.get_tesseract_version())

5.5.0.20241111


### Library

In [4]:
# Import the requests library to send HTTP requests and fetch content from websites
import requests

# Import BeautifulSoup for parsing HTML content easily
from bs4 import BeautifulSoup

# Import csv module to write and manage CSV files
import csv

# Import re for using regular expressions in text cleaning
import re

# Import Image module from PIL (Python Imaging Library) to handle image data
from PIL import Image

# Import BytesIO to handle byte streams (used to load image content from URLs)
from io import BytesIO

# Import BLIP model and processor from Hugging Face Transformers
#   - BlipProcessor prepares the image and text for the model
#   - BlipForConditionalGeneration is the BLIP model for generating image captions
from transformers import BlipProcessor, BlipForConditionalGeneration


d:\python3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load BLIP model
print("Loading BLIP model...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
print("BLIP model loaded.\n")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading BLIP model...
BLIP model loaded.



In [6]:
# Scrape the website
url = "https://www.magicalmelghat.in"
print(f"Fetching images from: {url}")
res = requests.get(url)
soup = BeautifulSoup(res.text, "html.parser")

Fetching images from: https://www.magicalmelghat.in


In [7]:
# Prepare CSV
csv_data = [["title", "url", "alt text", "missing alt", "blip caption", "ocr", "ocr_result", "result"]]

In [8]:
# Find images
img_tags = soup.find_all("img")
print(f"Found {len(img_tags)} image(s). Processing...\n")

Found 105 image(s). Processing...



In [9]:
# Loop through all image tags found on the webpage
for index, img_tag in enumerate(img_tags, start=1):
    img_url = img_tag.get("src")           # Get the image source URL
    alt = img_tag.get("alt")               # Get the alt attribute (alternative text)

    if not img_url:
        continue  # Skip if image URL is missing

    # Ensure the image URL is absolute (i.e., includes full domain)
    if not img_url.startswith("http"):
        img_url = url + "/" + img_url.lstrip("/")

    print(f"[{index}/{len(img_tags)}] Processing image: {img_url}")

    try:
        # Download the image from the web
        img_response = requests.get(img_url)

        # Open the image using PIL and convert to RGB mode
        img = Image.open(BytesIO(img_response.content)).convert("RGB")

        # Extract text using OCR (Tesseract)
        extracted_text = pytesseract.image_to_string(img)

        # Clean the extracted OCR text
        cleaned_text = ' '.join(extracted_text.splitlines())          # Merge lines
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text)              # Remove extra whitespace
        cleaned_text = re.sub(r'[^\w\s,]', '', cleaned_text)          # Remove unwanted punctuation

        # Generate a caption using BLIP model
        inputs = processor(images=img, return_tensors="pt")           # Preprocess image for BLIP
        out = model.generate(**inputs)                                # Generate caption tokens
        blip_caption = processor.decode(out[0], skip_special_tokens=True)  # Decode caption

        # Combine OCR result and BLIP caption (if OCR text exists)
        result = f"{cleaned_text} {blip_caption}".strip() if cleaned_text else blip_caption

        # Add collected data to CSV structure
        csv_data.append([
            alt or "",                    # Title: use alt text if available
            img_url,                      # Image URL
            alt or "",                    # Alt text (again)
            "yes" if not alt else "no",   # Missing alt text?
            blip_caption,                 # BLIP-generated caption
            "yes" if cleaned_text else "no",  # Was OCR text found?
            cleaned_text,                # OCR extracted text
            result                        # Final combined result
        ])

    except Exception as e:
        # Print error if any image fails to process
        print(f"Error processing image {img_url}: {e}")


[1/105] Processing image: https://www.magicalmelghat.in/public/website/image/jointlogonew.png
[2/105] Processing image: https://www.magicalmelghat.in/public/website/image/melghatlogo-name.png
[3/105] Processing image: https://www.magicalmelghat.in/public/website/image/logo.png
[4/105] Processing image: https://www.magicalmelghat.in/public/website/image/close.png


d:\python3.12\Lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[5/105] Processing image: https://www.magicalmelghat.in/public/website/image/satymevlogo.png
[6/105] Processing image: https://www.magicalmelghat.in/public/website/image/mh-logo.png
[7/105] Processing image: https://magicalmelghat.in/public/images/banner/1585136941hero1.png
[8/105] Processing image: https://magicalmelghat.in/public/images/banner/1585136982hero2.png
[9/105] Processing image: https://magicalmelghat.in/public/images/banner/1585136983hero3.jpg
[10/105] Processing image: https://www.magicalmelghat.in/public/website/image/meghat-paces/melghat-image1.jpeg
[11/105] Processing image: https://www.magicalmelghat.in/public/website/image/meghat-paces/melghat-image5.jpg
[12/105] Processing image: https://www.magicalmelghat.in/public/website/image/meghat-paces/melghat-image4.jpg
[13/105] Processing image: https://www.magicalmelghat.in/public/website/image/meghat-paces/melghat-image8.jpg
[14/105] Processing image: https://www.magicalmelghat.in/public/website/image/meghat-paces/melghat

In [10]:
# Save to CSV
output_file = "blip_ocr_report_result_view_imgs.csv"
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)

print(f"\n📄 CSV created: {output_file}")


📄 CSV created: blip_ocr_report_result_view_imgs.csv
